## 1. Imports & Setup

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.linear_model import (
    LinearRegression, LogisticRegression,
)

from sklearn.metrics import (
    r2_score, mean_absolute_error, root_mean_squared_error,
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
import os

RESULTS_DIR = "results/lasso_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Results directory:", RESULTS_DIR)

Results directory: results/lasso_results


In [4]:
RESULTS_DIR = "results/lasso_results" # Create this directory

## 2. Load Data

In [5]:
df = pd.read_csv("cibil_score.csv")
df = df.drop(columns=["Unnamed: 0"])

# normalize column names
df.columns = [col.lower().strip() for col in df.columns]

print(df.shape)
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.head()

(51336, 87)
Duplicate rows: 0


,prospectid,total_tl,tot_closed_tl,tot_active_tl,total_tl_opened_l6m,tot_tl_closed_l6m,pct_tl_open_l6m,pct_tl_closed_l6m,pct_active_tl,pct_closed_tl,...,pct_cc_enq_l6m_of_l12m,pct_pl_enq_l6m_of_ever,pct_cc_enq_l6m_of_ever,max_unsec_exposure_inpct,hl_flag,gl_flag,last_prod_enq2,first_prod_enq2,credit_score,approved_flag
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


In [6]:
df["approved_flag"].value_counts()

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64

## 3. Feature / Target Setup
* `X` -> all columns except `credit_score` and `approved_flag`
* `y_linear` -> `credit_score` (Linear Regression target)
* `y_multiclass` -> `approved_flag` (Logistic Regression, multiclass: P1/P2/P3/P4)
* `y_binary` -> `approved_flag` mapped to **1** for P1/P2 and **0** for P3/P4 (Logistic Regression, binary)

In [7]:
X = df.drop(columns=["approved_flag", "credit_score"])
y_linear = df["credit_score"].astype(float)
y_multiclass = df["approved_flag"].astype(str)

binary_map = {"P1": 1, "P2": 1, "P3": 0, "P4": 0}
y_binary = df["approved_flag"].map(binary_map)

assert y_binary.isna().sum() == 0, "approved_flag has values outside P1-P4"

print(y_multiclass.value_counts())
print(y_binary.value_counts())

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64
approved_flag
1    38002
0    13334
Name: count, dtype: int64


## 4. Train / Test Split
A single split on `X` is created and then reused (via the shared index) for every target so that every model family sees the exact same rows in train/test.

In [8]:
X_train, X_test, idx_train, idx_test = train_test_split(
    X, X.index, test_size=0.2, random_state=RANDOM_STATE
)

y_linear_train, y_linear_test = y_linear.loc[idx_train], y_linear.loc[idx_test]
y_multi_train, y_multi_test = y_multiclass.loc[idx_train], y_multiclass.loc[idx_test]
y_bin_train, y_bin_test = y_binary.loc[idx_train], y_binary.loc[idx_test]

print(X_train.shape, X_test.shape)

(41068, 85) (10268, 85)


## 5. Missing Value Handling
Domain-specific cleanup carried over/expanded from the original notebook. The dataset encodes several kinds of \"missing\" as the sentinel value `-99999`; the correct treatment depends on what the field means.

In [9]:
# 5a. Drop columns with very high missingness
high_missing = [c for c in ["cc_utilization", "pl_utilization"] if c in X_train.columns]
X_train = X_train.drop(columns=high_missing)
X_test = X_test.drop(columns=high_missing)

# 5b. Median-impute a handful of numeric fields where -99999 means "unknown"
median_columns = [
    "age_oldest_tl", "age_newest_tl", "pct_currentbal_all_tl", "time_since_recent_payment",
]
for col in median_columns:
    if col not in X_train.columns:
        continue
    X_train[col] = X_train[col].replace(-99999, np.nan)
    X_test[col] = X_test[col].replace(-99999, np.nan)
    train_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)  # use TRAIN median to avoid leakage

# 5c. Delinquency fields: -99999 means "never delinquent" -> 0
delinquency_columns = [
    "max_delinquency_level", "max_deliq_6mts", "max_deliq_12mts",
    "time_since_recent_deliquency", "time_since_first_deliquency",
]
delinquency_columns = [c for c in delinquency_columns if c in X_train.columns]
X_train[delinquency_columns] = X_train[delinquency_columns].replace(-99999, 0)
X_test[delinquency_columns] = X_test[delinquency_columns].replace(-99999, 0)

# 5d. Enquiry fields: -99999 means "no enquiry" -> 0
enquiry_columns = [
    "tot_enq", "cc_enq", "pl_enq", "cc_enq_l6m", "cc_enq_l12m",
    "pl_enq_l6m", "pl_enq_l12m", "enq_l3m", "enq_l6m", "enq_l12m",
]
enquiry_columns = [c for c in enquiry_columns if c in X_train.columns]
X_train[enquiry_columns] = X_train[enquiry_columns].replace(-99999, 0)
X_test[enquiry_columns] = X_test[enquiry_columns].replace(-99999, 0)

# 5e. time_since_recent_enq: -99999 = "no enquiry ever" -> longer than any observed gap,
# keep the "no enquiry" signal as its own binary flag (fit on TRAIN only, applied to both)
col = "time_since_recent_enq"
if col in X_train.columns:
    train_mask = X_train[col] == -99999
    test_mask = X_test[col] == -99999
    fill_value = X_train.loc[~train_mask, col].max() + 1

    X_train["no_enquiry_flag"] = train_mask.astype(int)
    X_test["no_enquiry_flag"] = test_mask.astype(int)

    X_train[col] = X_train[col].replace(-99999, fill_value)
    X_test[col] = X_test[col].replace(-99999, fill_value)

# 5f. max_unsec_exposure_inpct: -99999 = "no unsecured loan" -> 0%
col = "max_unsec_exposure_inpct"
if col in X_train.columns:
    X_train[col] = X_train[col].replace(-99999, 0)
    X_test[col] = X_test[col].replace(-99999, 0)

# 5g. Log-transform skewed income
if "netmonthlyincome" in X_train.columns:
    X_train["netmonthlyincome_log"] = np.log1p(X_train["netmonthlyincome"].clip(lower=0))
    X_test["netmonthlyincome_log"] = np.log1p(X_test["netmonthlyincome"].clip(lower=0))
    X_train = X_train.drop(columns=["netmonthlyincome"])
    X_test = X_test.drop(columns=["netmonthlyincome"])

print("Remaining NaNs in X_train:", X_train.isna().sum().sum())
print("Remaining NaNs in X_test:", X_test.isna().sum().sum())

Remaining NaNs in X_train: 0
Remaining NaNs in X_test: 0


## 6. Preprocessing Pipeline (scale numeric, one-hot encode categorical)

In [10]:
numeric_columns = X_train.select_dtypes(include=np.number).columns
categorical_columns = X_train.select_dtypes(include="object").columns
print(f"{len(numeric_columns)} numeric columns, {len(categorical_columns)} categorical columns")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Processed shapes:", X_train_processed.shape, X_test_processed.shape)

C:\Users\Kamal\AppData\Local\Temp\ipykernel_15364\2862013811.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(include="object").columns


79 numeric columns, 5 categorical columns
Processed shapes: (41068, 102) (10268, 102)


In [11]:
def evaluate_regression(model, X_te, y_te):
    y_pred = model.predict(X_te)
    return {
        "r2": r2_score(y_te, y_pred),
        "mae": mean_absolute_error(y_te, y_pred),
        "rmse": root_mean_squared_error(y_te, y_pred),
    }

def evaluate_classification(model, X_te, y_te, binary):
    y_pred = model.predict(X_te)
    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),
        "f1_macro": f1_score(y_te, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_te, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_te, y_pred, average="macro", zero_division=0),
        # Weighted averages
        "precision_weighted": precision_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        # Confusion Matrix
        "confusion_matrix": confusion_matrix(y_te, y_pred)        
    }
    if binary and hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_te, y_prob)
    return metrics

In [12]:
feature_names

array(['num__prospectid', 'num__total_tl', 'num__tot_closed_tl',
       'num__tot_active_tl', 'num__total_tl_opened_l6m',
       'num__tot_tl_closed_l6m', 'num__pct_tl_open_l6m',
       'num__pct_tl_closed_l6m', 'num__pct_active_tl',
       'num__pct_closed_tl', 'num__total_tl_opened_l12m',
       'num__tot_tl_closed_l12m', 'num__pct_tl_open_l12m',
       'num__pct_tl_closed_l12m', 'num__tot_missed_pmnt', 'num__auto_tl',
       'num__cc_tl', 'num__consumer_tl', 'num__gold_tl', 'num__home_tl',
       'num__pl_tl', 'num__secured_tl', 'num__unsecured_tl',
       'num__other_tl', 'num__age_oldest_tl', 'num__age_newest_tl',
       'num__time_since_recent_payment',
       'num__time_since_first_deliquency',
       'num__time_since_recent_deliquency', 'num__num_times_delinquent',
       'num__max_delinquency_level', 'num__max_recent_level_of_deliq',
       'num__num_deliq_6mts', 'num__num_deliq_12mts',
       'num__num_deliq_6_12mts', 'num__max_deliq_6mts',
       'num__max_deliq_12mts', 'num

## Least-Important Features via Lasso
Lasso's L1 penalty drives the coefficients of uninformative features to exactly zero, which makes it a built-in feature-selection tool: sort by `|coefficient|` and whatever sits near/at zero is what Lasso considers least useful for predicting the target. This is done for the **linear** target (`credit_score`, via `LassoCV`) and for **both logistic** targets (`approved_flag` binary & multiclass, via L1-penalized `LogisticRegressionCV`), since Lasso applies equally to regression and classification.

In [13]:
def get_lasso_ranked_features(coefs, feature_names, zero_tol=1e-4):
    """
    Return features ranked from least important to most important
    based on the absolute value of their Lasso coefficients.

    zeroed_out = True means the coefficient is effectively zero.
    """

    coefs = np.asarray(coefs)

    # For multiclass, coefs has shape:
    # (number_of_classes, number_of_features)
    # Summarize each feature using its L2 norm across classes.
    if coefs.ndim == 1:
        importance = np.abs(coefs)
        coefficient = coefs

    elif coefs.ndim == 2:
        importance = np.linalg.norm(coefs, axis=0)

        # Keep the coefficient information for reference
        coefficient = importance

    else:
        raise ValueError("Coefficients must be 1D or 2D.")

    ranked = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficient,
        "abs_coefficient": importance
    })

    ranked["zeroed_out"] = (
        ranked["abs_coefficient"] <= zero_tol
    )

    # Least important first
    ranked = ranked.sort_values(
        "abs_coefficient",
        ascending=True
    ).reset_index(drop=True)

    return ranked

### Get least important features for Linear regression and save then to 'lasso_least_important_features_linear.csv'

In [14]:
# --- Linear regression (credit_score): LassoCV auto-selects alpha via internal CV ---
lasso_cv_linear = LassoCV() # put input parameters
# code to fit and get ranked features
linear_lasso_ranked = pd.DataFrame() # change to save features
linear_lasso_ranked.to_csv(f"{RESULTS_DIR}/lasso_least_important_features_linear.csv", index=False)
linear_lasso_ranked.head(15)  # 15 LEAST important features

""


In [15]:
# --- Linear regression (credit_score) ---

lasso_cv_linear = LassoCV(
    cv=5,
    random_state=RANDOM_STATE,
    max_iter=10000,
    n_jobs=-1
)

# Fit Lasso
lasso_cv_linear.fit(
    X_train_processed,
    y_linear_train
)

print("Best alpha:", lasso_cv_linear.alpha_)

# Rank features
linear_lasso_ranked = get_lasso_ranked_features(
    lasso_cv_linear.coef_,
    feature_names
)

# Save results
linear_lasso_ranked.to_csv(
    f"{RESULTS_DIR}/lasso_least_important_features_linear.csv",
    index=False
)

# Display 15 least important features
linear_lasso_ranked.head(15)

Best alpha: 0.012194521696249221


,feature,coefficient,abs_coefficient,zeroed_out
0,num__total_tl,0.0,0.0,True
1,num__tot_closed_tl,0.0,0.0,True
2,num__tot_active_tl,0.0,0.0,True
3,num__pct_tl_closed_l6m,-0.0,0.0,True
4,num__tot_tl_closed_l12m,-0.0,0.0,True
5,num__total_tl_opened_l12m,0.0,0.0,True
6,num__pct_closed_tl,-0.0,0.0,True
7,num__pct_active_tl,0.0,0.0,True
8,num__auto_tl,-0.0,0.0,True
9,num__tot_missed_pmnt,0.0,0.0,True


### Get selected features and apply Linear Regression

In [16]:
# Select features that Lasso did NOT zero out
selected_linear_features = linear_lasso_ranked.loc[
    ~linear_lasso_ranked["zeroed_out"],
    "feature"
].tolist()

print(
    "Original number of processed features:",
    X_train_processed.shape[1]
)

print(
    "Selected features:",
    len(selected_linear_features)
)

print(
    "Removed features:",
    X_train_processed.shape[1] -
    len(selected_linear_features)
)

# Reduced datasets
X_train_linear_selected = X_train_processed[
    selected_linear_features
]

X_test_linear_selected = X_test_processed[
    selected_linear_features
]

# Train Linear Regression using selected features
linear_selected = LinearRegression()

linear_selected.fit(
    X_train_linear_selected,
    y_linear_train
)

# Predictions
y_pred_linear_selected = linear_selected.predict(
    X_test_linear_selected
)

# Metrics
r2_selected = r2_score(
    y_linear_test,
    y_pred_linear_selected
)

mae_selected = mean_absolute_error(
    y_linear_test,
    y_pred_linear_selected
)

rmse_selected = root_mean_squared_error(
    y_linear_test,
    y_pred_linear_selected
)

print("\nLinear Regression after Lasso Feature Selection")
print("-----------------------------------------------")
print("R²   :", r2_selected)
print("MAE  :", mae_selected)
print("RMSE :", rmse_selected)

Original number of processed features: 102
Selected features: 50
Removed features: 52

Linear Regression after Lasso Feature Selection
-----------------------------------------------
R²   : 0.9106074026831212
MAE  : 5.3182844933693785
RMSE : 6.114924658828916


### Get least important features for Logistic Regression for binary classification and save then to 'lasso_least_important_features_logistic_binary.csv'

In [17]:
# --- Logistic regression, binary (approved_flag P1/P2 vs P3/P4): L1 LogisticRegressionCV ---
logreg_cv_binary = LogisticRegressionCV() # put input parameters
# code to fit and get ranked features
binary_lasso_ranked = pd.DataFrame() # change to save features
binary_lasso_ranked.to_csv(f"{RESULTS_DIR}/lasso_least_important_features_logistic_binary.csv", index=False)
binary_lasso_ranked.head(15)

""


In [18]:
# --- Logistic regression, binary ---
# P1/P2 = 1
# P3/P4 = 0

logreg_cv_binary = LogisticRegressionCV(
    Cs=10,
    cv=5,
    penalty="l1",
    solver="liblinear",
    scoring="f1",
    max_iter=5000,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Fit
logreg_cv_binary.fit(
    X_train_processed,
    y_bin_train
)

print("Best C:", logreg_cv_binary.C_[0])

# Rank features
binary_lasso_ranked = get_lasso_ranked_features(
    logreg_cv_binary.coef_,
    feature_names
)

# Save
binary_lasso_ranked.to_csv(
    f"{RESULTS_DIR}/lasso_least_important_features_logistic_binary.csv",
    index=False
)

# Show 15 least important features
binary_lasso_ranked.head(15)

C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2123: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\line

Best C: 0.046415888336127774


,feature,coefficient,abs_coefficient,zeroed_out
0,num__total_tl,0.0,0.0,True
1,num__tot_closed_tl,0.0,0.0,True
2,num__tot_active_tl,0.0,0.0,True
3,num__tot_tl_closed_l12m,0.0,0.0,True
4,num__total_tl_opened_l12m,0.0,0.0,True
5,num__home_tl,0.0,0.0,True
6,num__consumer_tl,0.0,0.0,True
7,num__cc_tl,0.0,0.0,True
8,num__pl_tl,0.0,0.0,True
9,num__max_recent_level_of_deliq,0.0,0.0,True


### Get selected features and apply Logistic Regression for Binary Classification

In [19]:
# Select features retained by L1 Logistic Regression
selected_binary_features = binary_lasso_ranked.loc[
    ~binary_lasso_ranked["zeroed_out"],
    "feature"
].tolist()

print(
    "Original number of processed features:",
    X_train_processed.shape[1]
)

print(
    "Selected features:",
    len(selected_binary_features)
)

print(
    "Removed features:",
    X_train_processed.shape[1] -
    len(selected_binary_features)
)

# Reduced datasets
X_train_binary_selected = X_train_processed[
    selected_binary_features
]

X_test_binary_selected = X_test_processed[
    selected_binary_features
]

# Train logistic regression
binary_selected_model = LogisticRegression(
    penalty="l2",
    max_iter=5000,
    random_state=RANDOM_STATE
)

binary_selected_model.fit(
    X_train_binary_selected,
    y_bin_train
)

# Prediction
y_pred_binary_selected = binary_selected_model.predict(
    X_test_binary_selected
)

# Metrics
accuracy_binary_selected = accuracy_score(
    y_bin_test,
    y_pred_binary_selected
)

precision_binary_selected = precision_score(
    y_bin_test,
    y_pred_binary_selected,
    zero_division=0
)

recall_binary_selected = recall_score(
    y_bin_test,
    y_pred_binary_selected,
    zero_division=0
)

f1_binary_selected = f1_score(
    y_bin_test,
    y_pred_binary_selected,
    zero_division=0
)

print("\nBinary Logistic Regression after L1 Feature Selection")
print("------------------------------------------------------")
print("Accuracy :", accuracy_binary_selected)
print("Precision:", precision_binary_selected)
print("Recall   :", recall_binary_selected)
print("F1 Score :", f1_binary_selected)

Original number of processed features: 102
Selected features: 59
Removed features: 43


C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(



Binary Logistic Regression after L1 Feature Selection
------------------------------------------------------
Accuracy : 0.8934553954031944
Precision: 0.9060207437389325
Recall   : 0.9532871972318339
F1 Score : 0.92905317769131


### Get least important features for Logistic Regression for multiclass classification and save then to 'lasso_least_important_features_logistic_multiclass.csv'

In [20]:
# --- Logistic regression, multiclass (P1/P2/P3/P4): one coefficient vector per class,
# so a feature's overall importance is summarized as the L2 norm across classes ---
logreg_cv_multi = LogisticRegressionCV() # put input parameters
# code to fit and get ranked features
multi_lasso_ranked = pd.DataFrame()
multi_lasso_ranked.to_csv(f"{RESULTS_DIR}/lasso_least_important_features_logistic_multiclass.csv", index=False)
multi_lasso_ranked.head(15)

""


In [21]:
# --- Logistic regression, multiclass ---
# P1 / P2 / P3 / P4

logreg_cv_multi = LogisticRegressionCV(
    Cs=10,
    cv=5,
    penalty="l1",
    solver="saga",
    scoring="f1_macro",
    max_iter=10000,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Fit
logreg_cv_multi.fit(
    X_train_processed,
    y_multi_train
)

print("Best C values:", logreg_cv_multi.C_)

# Rank features
multi_lasso_ranked = get_lasso_ranked_features(
    logreg_cv_multi.coef_,
    feature_names
)

# Save
multi_lasso_ranked.to_csv(
    f"{RESULTS_DIR}/lasso_least_important_features_logistic_multiclass.csv",
    index=False
)

# Display 15 least important features
multi_lasso_ranked.head(15)

C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2123: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
C:\Users\Kamal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\line

Best C values: [2.7825594 2.7825594 2.7825594 2.7825594]


,feature,coefficient,abs_coefficient,zeroed_out
0,num__total_tl,0.010356,0.010356,False
1,num__hl_flag,0.010461,0.010461,False
2,num__tot_closed_tl,0.010616,0.010616,False
3,cat__first_prod_enq2_CC,0.013656,0.013656,False
4,num__num_sub_12mts,0.022966,0.022966,False
5,num__gold_tl,0.022989,0.022989,False
6,num__other_tl,0.023834,0.023834,False
7,num__pct_active_tl,0.027837,0.027837,False
8,num__pct_of_active_tls_ever,0.027837,0.027837,False
9,num__pct_closed_tl,0.027837,0.027837,False


### Get selected features and apply Logistic Regression for Multiclass Classification

In [22]:
# Select features retained by L1 Logistic Regression
selected_multi_features = multi_lasso_ranked.loc[
    ~multi_lasso_ranked["zeroed_out"],
    "feature"
].tolist()

print(
    "Original number of processed features:",
    X_train_processed.shape[1]
)

print(
    "Selected features:",
    len(selected_multi_features)
)

print(
    "Removed features:",
    X_train_processed.shape[1] -
    len(selected_multi_features)
)

# Reduced datasets
X_train_multi_selected = X_train_processed[
    selected_multi_features
]

X_test_multi_selected = X_test_processed[
    selected_multi_features
]

# Train multiclass Logistic Regression
multi_selected_model = LogisticRegression(
    max_iter=10000,
    random_state=RANDOM_STATE
)

multi_selected_model.fit(
    X_train_multi_selected,
    y_multi_train
)

# Predictions
y_pred_multi_selected = multi_selected_model.predict(
    X_test_multi_selected
)

# Metrics
accuracy_multi_selected = accuracy_score(
    y_multi_test,
    y_pred_multi_selected
)

f1_macro_multi_selected = f1_score(
    y_multi_test,
    y_pred_multi_selected,
    average="macro",
    zero_division=0
)

precision_macro_multi_selected = precision_score(
    y_multi_test,
    y_pred_multi_selected,
    average="macro",
    zero_division=0
)

recall_macro_multi_selected = recall_score(
    y_multi_test,
    y_pred_multi_selected,
    average="macro",
    zero_division=0
)

print("\nMulticlass Logistic Regression after L1 Feature Selection")
print("---------------------------------------------------------")
print("Accuracy       :", accuracy_multi_selected)
print("Precision Macro:", precision_macro_multi_selected)
print("Recall Macro   :", recall_macro_multi_selected)
print("F1 Macro       :", f1_macro_multi_selected)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_multi_test,
        y_pred_multi_selected
    )
)

Original number of processed features: 102
Selected features: 102
Removed features: 0

Multiclass Logistic Regression after L1 Feature Selection
---------------------------------------------------------
Accuracy       : 0.8031749123490456
Precision Macro: 0.7348751019765251
Recall Macro   : 0.6933865081468648
F1 Macro       : 0.7075830281575274

Confusion Matrix:
[[ 898  237    0    0]
 [ 133 5989  245   12]
 [  26  834  430  246]
 [   0   34  254  930]]


In [23]:
# --- Combined view: features Lasso considers unimportant across all three targets ---
common_unimportant = set(linear_lasso_ranked.loc[linear_lasso_ranked["zeroed_out"], "feature"]) \
    & set(binary_lasso_ranked.loc[binary_lasso_ranked["zeroed_out"], "feature"]) \
    & set(multi_lasso_ranked.loc[multi_lasso_ranked["zeroed_out"], "feature"])

print(f"{len(common_unimportant)} features zeroed out by Lasso in ALL THREE models "
      f"(safe candidates to drop):")
sorted(common_unimportant)

0 features zeroed out by Lasso in ALL THREE models (safe candidates to drop):


[]

In [24]:
common_unimportant = (
    set(linear_lasso_ranked.loc[
        linear_lasso_ranked["zeroed_out"], "feature"
    ])
    &
    set(binary_lasso_ranked.loc[
        binary_lasso_ranked["zeroed_out"], "feature"
    ])
    &
    set(multi_lasso_ranked.loc[
        multi_lasso_ranked["zeroed_out"], "feature"
    ])
)

print(
    f"{len(common_unimportant)} features zeroed out by "
    f"Lasso in ALL THREE models "
    f"(safe candidates to drop):"
)

sorted(common_unimportant)

0 features zeroed out by Lasso in ALL THREE models (safe candidates to drop):


[]

In [25]:
print("Linear zeroed:",
      linear_lasso_ranked["zeroed_out"].sum())

print("Binary zeroed:",
      binary_lasso_ranked["zeroed_out"].sum())

print("Multiclass zeroed:",
      multi_lasso_ranked["zeroed_out"].sum())

Linear zeroed: 52
Binary zeroed: 43
Multiclass zeroed: 0


In [26]:
linear_zeroed = set(
    linear_lasso_ranked.loc[
        linear_lasso_ranked["zeroed_out"],
        "feature"
    ]
)

binary_zeroed = set(
    binary_lasso_ranked.loc[
        binary_lasso_ranked["zeroed_out"],
        "feature"
    ]
)

multi_zeroed = set(
    multi_lasso_ranked.loc[
        multi_lasso_ranked["zeroed_out"],
        "feature"
    ]
)

print("\nLinear only:")
print(sorted(linear_zeroed))

print("\nBinary only:")
print(sorted(binary_zeroed))

print("\nMulticlass only:")
print(sorted(multi_zeroed))


Linear only:
['cat__education_12TH', 'cat__education_GRADUATE', 'cat__education_OTHERS', 'cat__education_POST-GRADUATE', 'cat__education_PROFESSIONAL', 'cat__education_SSC', 'cat__first_prod_enq2_AL', 'cat__first_prod_enq2_CC', 'cat__first_prod_enq2_HL', 'cat__first_prod_enq2_PL', 'cat__first_prod_enq2_others', 'cat__gender_F', 'cat__gender_M', 'cat__last_prod_enq2_AL', 'cat__last_prod_enq2_CC', 'cat__last_prod_enq2_ConsumerLoan', 'cat__last_prod_enq2_HL', 'cat__last_prod_enq2_others', 'cat__maritalstatus_Married', 'cat__maritalstatus_Single', 'num__auto_tl', 'num__cc_enq_l12m', 'num__cc_flag', 'num__cc_tl', 'num__enq_l12m', 'num__gl_flag', 'num__home_tl', 'num__max_recent_level_of_deliq', 'num__num_dbt', 'num__num_dbt_6mts', 'num__num_deliq_12mts', 'num__num_deliq_6_12mts', 'num__num_lss_6mts', 'num__num_sub', 'num__num_times_60p_dpd', 'num__pct_active_tl', 'num__pct_cc_enq_l6m_of_l12m', 'num__pct_closed_tl', 'num__pct_of_active_tls_ever', 'num__pct_opened_tls_l6m_of_l12m', 'num__pct

In [27]:
all_zeroed = (
    linear_zeroed
    | binary_zeroed
    | multi_zeroed
)

print(
    f"{len(all_zeroed)} features were zeroed out "
    "in at least one model:"
)

print(sorted(all_zeroed))

68 features were zeroed out in at least one model:
['cat__education_12TH', 'cat__education_GRADUATE', 'cat__education_OTHERS', 'cat__education_POST-GRADUATE', 'cat__education_PROFESSIONAL', 'cat__education_SSC', 'cat__education_UNDER GRADUATE', 'cat__first_prod_enq2_AL', 'cat__first_prod_enq2_CC', 'cat__first_prod_enq2_ConsumerLoan', 'cat__first_prod_enq2_HL', 'cat__first_prod_enq2_PL', 'cat__first_prod_enq2_others', 'cat__gender_F', 'cat__gender_M', 'cat__last_prod_enq2_AL', 'cat__last_prod_enq2_CC', 'cat__last_prod_enq2_ConsumerLoan', 'cat__last_prod_enq2_HL', 'cat__last_prod_enq2_PL', 'cat__last_prod_enq2_others', 'cat__maritalstatus_Married', 'cat__maritalstatus_Single', 'num__auto_tl', 'num__cc_enq', 'num__cc_enq_l12m', 'num__cc_flag', 'num__cc_tl', 'num__consumer_tl', 'num__enq_l12m', 'num__gl_flag', 'num__home_tl', 'num__max_recent_level_of_deliq', 'num__num_dbt', 'num__num_dbt_12mts', 'num__num_dbt_6mts', 'num__num_deliq_12mts', 'num__num_deliq_6_12mts', 'num__num_lss', 'num__n

In [28]:
print(linear_lasso_ranked[linear_lasso_ranked["zeroed_out"]])
print(binary_lasso_ranked[binary_lasso_ranked["zeroed_out"]])
print(multi_lasso_ranked[multi_lasso_ranked["zeroed_out"]])

                             feature  coefficient  abs_coefficient  zeroed_out
0                      num__total_tl          0.0              0.0        True
1                 num__tot_closed_tl          0.0              0.0        True
2                 num__tot_active_tl          0.0              0.0        True
3             num__pct_tl_closed_l6m         -0.0              0.0        True
4            num__tot_tl_closed_l12m         -0.0              0.0        True
5          num__total_tl_opened_l12m          0.0              0.0        True
6                 num__pct_closed_tl         -0.0              0.0        True
7                 num__pct_active_tl          0.0              0.0        True
8                       num__auto_tl         -0.0              0.0        True
9               num__tot_missed_pmnt          0.0              0.0        True
10  num__time_since_first_deliquency         -0.0              0.0        True
11    num__max_recent_level_of_deliq          0.0   